# DTR Navigation in Windy Mini Grid Environment

This notebook demonstrates how to use the Dynamic Treatment Regime (DTR) approach to navigate through a windy grid world environment. We'll use the ExtendedWindyGridSCM class that integrates DTR with the mini-grid environment.

## Setup and Imports

In [18]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium.utils.play import play
import minigrid
from minigrid.wrappers import ImgObsWrapper

# Import causal gym components
from causal_gym.envs import CustomCrossingEnv
from causal_gym.envs import WindyMiniGridPCH
from causal_gym.envs.extended_windy_grid import ExtendedWindyGridPCH

# Import DTR model
from causal_gym.envs.dtr_example import DTRExamplePCH

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)

## Creating the Environment

We'll create a custom lava crossing environment and wrap it with our extended windy grid wrapper that integrates DTR.


In [19]:
# Create the base environment
lavagrid = gym.make('MiniGrid-LavaGapS6-v0', max_episode_steps=30, 
                   agent_pov=False, render_mode='rgb_array', 
                   highlight=False, tile_size=32)

# Create the extended windy grid with DTR decision model
ext_windy_lavagrid = ExtendedWindyGridPCH(
    env=lavagrid, 
    decision_type='DTR',
    show_wind=True, 
    wind_dist=[.2, .2, .2, .2, .2]  # Equal probability for all wind directions
)

# Reset the environment
observation, info = ext_windy_lavagrid.reset(seed=SEED)
print(f"Initial State (S1): {observation}")
print(f"Initial Wind Direction: {info['wind']}")

Initial State (S1): (1, 1)
Initial Wind Direction: 3


## Single Pass Demonstration

Let's first demonstrate a single pass with a few steps, showing the state transitions in the DTR model.


In [20]:
# Define simple behavioral policies for the DTR model
def behavior_policy1(s1):
    """First stage policy (S1 -> X1)"""
    # For demonstration, just move right if state is 0, otherwise move forward
    return 1 if s1 == 0 else 2

def behavior_policy2(s1, x1, s2):
    """Second stage policy (S1, X1, S2 -> X2)"""
    # Simple policy based on S2 state
    return 0 if s2 == 0 else 1  # left if S2=0, right if S2=1

# Reset for demonstration
observation, info = ext_windy_lavagrid.reset(seed=SEED)
init_obs = ext_windy_lavagrid.render()

# First step - use 'see' with our behavioral policy
action, next_state, reward, terminated, truncated, info = ext_windy_lavagrid.see(
    bpolicy=lambda state, wind: ext_windy_lavagrid.env._map_decision_to_grid_action(behavior_policy1(observation))
)

# Save the S2 state
s2_observation = next_state

print(f"\nStep 1:")
print(f"Action (X1): {action}")
print(f"Next State (S2): {next_state}")
print(f"Reward: {reward}")
print(f"Wind Direction: {info['wind']}")
print(f"DTR Stage: {info.get('dtr_stage', 'N/A')}")
step1_obs = ext_windy_lavagrid.render()

# Second step - now we can use s2_observation
action2, next_state2, reward2, terminated2, truncated2, info2 = ext_windy_lavagrid.see(
    bpolicy=lambda state, wind: ext_windy_lavagrid.env._map_decision_to_grid_action(
        behavior_policy2(observation, action, s2_observation)
    )
)
print(f"\nStep 2:")
print(f"Action (X2): {action}")
print(f"Next State: {next_state}")
print(f"Reward: {reward}")
print(f"Wind Direction: {info['wind']}")
print(f"DTR Stage: {info.get('dtr_stage', 'N/A')}")
step2_obs = ext_windy_lavagrid.render()

# Third step - continue navigation
action, next_state, reward, terminated, truncated, info = ext_windy_lavagrid.see()
print(f"\nStep 3:")
print(f"Action: {action}")
print(f"Next State: {next_state}")
print(f"Reward: {reward}")
print(f"Wind Direction: {info['wind']}")
print(f"DTR Complete: {info.get('dtr_complete', 'N/A')}")
step3_obs = ext_windy_lavagrid.render()

# Fourth step
action, next_state, reward, terminated, truncated, info = ext_windy_lavagrid.see()
print(f"\nStep 4:")
print(f"Action: {action}")
print(f"Next State: {next_state}")
print(f"Reward: {reward}")
print(f"Wind Direction: {info['wind']}")
print(f"DTR Complete: {info.get('dtr_complete', 'N/A')}")
step4_obs = ext_windy_lavagrid.render()

# Visualize the steps
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(init_obs)
axes[0].axis('off')
axes[0].set_title('Initial State (S1)')

axes[1].imshow(step1_obs)
axes[1].axis('off')
axes[1].set_title('After First Step (S2)')

axes[2].imshow(step2_obs)
axes[2].axis('off')
axes[2].set_title('After Second Step (DTR Complete)')

axes[3].imshow(step4_obs)
axes[3].axis('off')
axes[3].set_title('After Fourth Step')

plt.tight_layout()
plt.show()


Step 1:
Action (X1): 2
Next State (S2): (2, 1)
Reward: -1.1
Wind Direction: 2
DTR Stage: N/A


TypeError: unsupported operand type(s) for +: 'float' and 'function'

## Multiple Passes Through the Environment

Now, let's run 100 passes through the environment and collect statistics.

In [16]:
# Run 100 passes and collect data
num_passes = 100
results = []

for pass_idx in range(num_passes):
    # Reset environment for new pass
    observation, info = ext_windy_lavagrid.reset(seed=SEED + pass_idx)
    
    # Initialize tracking variables for this pass
    pass_rewards = 0
    steps_taken = 0
    reached_goal = False
    
    # Continue until terminated or max steps reached
    terminated = False
    truncated = False
    
    while not (terminated or truncated) and steps_taken < 30:  # 30 steps max per pass
        # Use behavioral policies through the see() method
        if steps_taken == 0:
            # First stage of DTR
            action, next_state, reward, terminated, truncated, info = ext_windy_lavagrid.see(
                bpolicy=lambda state, wind: ext_windy_lavagrid.env._map_decision_to_grid_action(
                    behavior_policy1(observation)
                )
            )
        elif steps_taken == 1:
            # Second stage of DTR
            action, next_state, reward, terminated, truncated, info = ext_windy_lavagrid.see(
                bpolicy=lambda state, wind: ext_windy_lavagrid.env._map_decision_to_grid_action(
                    behavior_policy2(observation, action, next_state)
                )
            )
        else:
            # Continue navigation with behavioral policy
            action, next_state, reward, terminated, truncated, info = ext_windy_lavagrid.see()
        
        # Update tracking
        observation = next_state
        pass_rewards += reward
        steps_taken += 1
        
        # Check if goal reached
        if terminated and reward > 0:
            reached_goal = True
    
    # Store results for this pass
    results.append({
        'pass_idx': pass_idx,
        'total_reward': pass_rewards,
        'steps_taken': steps_taken,
        'reached_goal': reached_goal
    })
    
    # Print progress
    if (pass_idx + 1) % 10 == 0:
        print(f"Completed {pass_idx + 1} passes")

# Convert results to numpy arrays for analysis
pass_indices = np.array([r['pass_idx'] for r in results])
rewards = np.array([r['total_reward'] for r in results])
steps = np.array([r['steps_taken'] for r in results])
goals_reached = np.array([r['reached_goal'] for r in results])

# Calculate statistics
avg_reward = np.mean(rewards)
avg_steps = np.mean(steps)
success_rate = np.mean(goals_reached) * 100

print(f"\nResults after {num_passes} passes:")
print(f"Average reward: {avg_reward:.2f}")
print(f"Average steps per pass: {avg_steps:.2f}")
print(f"Success rate: {success_rate:.2f}%")

## Analyzing the Results

Let's visualize the results of our 100 passes.

In [17]:
# Visualize rewards distribution
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.hist(rewards, bins=10, alpha=0.7)
plt.axvline(avg_reward, color='r', linestyle='dashed', linewidth=2, label=f'Mean: {avg_reward:.2f}')
plt.xlabel('Total Reward')
plt.ylabel('Frequency')
plt.title('Distribution of Rewards')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(pass_indices, rewards, alpha=0.7)
plt.axhline(avg_reward, color='r', linestyle='dashed', linewidth=2, label=f'Mean: {avg_reward:.2f}')
plt.xlabel('Pass Number')
plt.ylabel('Total Reward')
plt.title('Rewards by Pass')
plt.legend()

plt.tight_layout()
plt.show()

# Visualize steps and success
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.hist(steps, bins=range(1, 32), alpha=0.7)
plt.axvline(avg_steps, color='r', linestyle='dashed', linewidth=2, label=f'Mean: {avg_steps:.2f}')
plt.xlabel('Steps Taken')
plt.ylabel('Frequency')
plt.title('Distribution of Steps')
plt.legend()

plt.subplot(1, 2, 2)
success_indices = pass_indices[goals_reached]
failure_indices = pass_indices[~goals_reached]
success_rewards = rewards[goals_reached]
failure_rewards = rewards[~goals_reached]

plt.scatter(success_indices, success_rewards, color='green', alpha=0.7, label='Goal Reached')
plt.scatter(failure_indices, failure_rewards, color='red', alpha=0.7, label='Goal Not Reached')
plt.xlabel('Pass Number')
plt.ylabel('Total Reward')
plt.title('Success vs. Failure')
plt.legend()

plt.tight_layout()
plt.show()

Creating DTR environment...


TypeError: object of type 'NoneType' has no len()